# E1 · W3 · Kaggle run 03b — xuất dữ liệu YOLO (CPU)
**Mục đích:** xuất ảnh + nhãn YOLO train/val (ảnh khả kiến, stride 10, test không xuất) theo ADR-0004, dùng kết quả review v2 và kết quả chấm bảng xem mù (issue #16).

**Trước khi chạy:**
1. Tạo **phiên bản mới** của dataset `e1-w3-tools` có thêm `review_assignment_w3.csv` (Claude tạo sau khi review).
2. *Add Input*: `e1-svanstrom-raw`, `e1-w3-tools`. *Settings*: Accelerator **None**, Internet **On**.
3. Điền `COMMIT` và `PLUS1_POLICY` (theo `offset_review_score.json`: `offset0` hoặc `offset1`).

**Sau khi chạy:** *Save Version → Save & Run All (Commit)*. Tải **chỉ** `w3_export_evidence.zip` → thư mục kết quả run 03b. Dữ liệu được đóng gói thành **một file** `e1-yolo-v1.zip` (không nén lại JPEG) để Kaggle gắn input ổn định; file này để nguyên trên Kaggle, run 04/05 tự giải nén. Đặt tên notebook là `e1-w3-run03b` để dễ chọn ở run 04.

In [ ]:
COMMIT = ""                     # <-- SHA day du 40 ky tu
PLUS1_POLICY = "offset0"        # <-- theo ADR-0006: offset0 | offset1
DATASET_VERSION = "e1-yolo-v1"  # offset1 -> "e1-yolo-v2" (run 07)
REVIEW_FILE = "review_assignment_w3.csv"

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

assert PLUS1_POLICY in ("offset0", "offset1"), "PLUS1_POLICY phai la offset0 hoac offset1"

In [ ]:
!pip -q install mcos-decoder
import mcos_decoder; print("mcos-decoder OK")

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
TOOLS = {n: find(n) for n in ["w3_export_yolo.py", "video_split_v1.csv", "annotation_audit.csv", REVIEW_FILE]}
for n, p in TOOLS.items():
    print(f"{n:28s} sha256={sha(p)[:16]}  {p}")

In [ ]:
!{sys.executable} {TOOLS["w3_export_yolo.py"]} --data-root {DATA_ROOT} --plus1-policy {PLUS1_POLICY} --split-csv {TOOLS["video_split_v1.csv"]} --audit {TOOLS["annotation_audit.csv"]} --review {TOOLS[REVIEW_FILE]} --out /kaggle/working --commit {COMMIT} --pack-zip --dataset-version {DATASET_VERSION}

## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
ev = "/kaggle/working/w3_export_evidence"
st = json.load(open(f"{ev}/stage_status.json"))
print({k: v["status"] for k, v in st.items()})
card = json.load(open(f"{ev}/dataset_card.json"))
print("split_counts:", card["inputs"]["split_counts"]); print("videos:", card["videos_included"])
print("images:", card["images"], "| skipped:", card["frames_skipped"], "| problems:", card["problems"])
stats = pd.read_csv(f"{ev}/dataset_stats.csv"); print(stats.to_string(index=False))
ok = all(v["status"] == "OK" for v in st.values()) and not card["problems"]
ok &= card["inputs"]["split_counts"] == {"train": 454, "val": 98, "test": 98}
ok &= not os.path.exists(f"/kaggle/working/{DATASET_VERSION}/images/test")
zinfo = json.load(open(f"{ev}/dataset_zip.json")); print("zip:", zinfo)
ok &= os.path.exists(f"/kaggle/working/{DATASET_VERSION}.zip") and sha(f"/kaggle/working/{DATASET_VERSION}.zip") == zinfo["sha256"]
print("fingerprint:", card["content_fingerprint_sha256"][:16], "| trung ban goc d0ac7e23...:", card["content_fingerprint_sha256"].startswith("d0ac7e23"))
for sp in ("train", "val"):
    have = set(stats[(stats.split == sp) & (stats.with_bbox > 0)].class_name)
    miss = {"Airplane", "Bird", "Drone", "Helicopter"} - have
    if miss:
        print("THIEU LOP o", sp, ":", miss); ok = False
print("PASS" if ok else "FAIL — gui stage_status.json + dataset_card.json cho Claude")

In [ ]:
p = "/kaggle/working/w3_export_evidence.zip"; print(os.path.getsize(p) // 1024, "KB  sha256", sha(p))